## Part 1: Configuration de l'environnement

In [ ]:
# 2. Installer les packages
!pip install -q langchain langchain-community transformers sentencepiece accelerate

In [ ]:
# 3. (Facultatif) Vérifiez le matériel
!nvidia-smi || echo "CPU runtime"

## Part 2 : Chargez un petit modèle ouvert et construisez votre première LLMChain

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain import PromptTemplate, LLMChain
from langchain_community.llms import HuggingFacePipeline

# Choose a small model suitable for CPU
model_name = "google/flan-t5-small"

# 1. Charger le modèle + tokenizer avec Transformers
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

# 2. Enveloppez-le dans un pipeline HuggingFace
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50, # Limit output length for small models
    torch_dtype=torch.bfloat16,
    device=0 if torch.cuda.is_available() else -1 # Use GPU if available, else CPU
)

# 3. Enveloppez le pipeline et construisez HuggingFacePipeline à l'aide d'un PromptTemplate
llm = HuggingFacePipeline(pipeline=pipe)

# Define a prompt template for rewriting
template = "Réécrivez ce texte pour qu'il soit plus simple pour les débutants : {text}"
prompt = PromptTemplate(template=template, input_variables=["text"])

# Build the LLMChain
llm_chain = LLMChain(prompt=prompt, llm=llm)

print(f"Modèle chargé : {model_name}")

In [ ]:
# 4. Testez avec une invite de réécriture conviviale
text_to_rewrite = "L'apprentissage automatique est un sous-domaine de l'intelligence artificielle qui permet aux systèmes d'apprendre à partir de données, d'identifier des modèles et de prendre des décisions avec une intervention humaine minimale."

print("Texte original:")
print(text_to_rewrite)

print("\nTexte réécrit (plus simple):")
print(llm_chain.run(text_to_rewrite))

## Part 3 : Composer un pipeline simple en deux étapes avec LangChain Runnables

In [ ]:
from langchain.schema.runnable import RunnableSequence

# 1. Créez deux modèles d’invite : un pour le résumé, un pour la mise en puces.
summary_template = "Veuillez résumer le texte suivant en une seule phrase :\n\n{text}"
summary_prompt = PromptTemplate(template=summary_template, input_variables=["text"])

bullet_point_template = "Veuillez prendre le texte suivant et le convertir en 3 points de puce :\n\n{summary}"
bullet_point_prompt = PromptTemplate(template=bullet_point_template, input_variables=["summary"])

print("Modèles d'invite pour le résumé et les points de puce créés.")

In [ ]:
# 2. Réutilisez le même wrapper LLM de la partie 2.
# The 'llm' object is already defined from Part 2.

# 3. Enchaînez-les avec RunnableSequence
pipeline = RunnableSequence(first=summary_prompt | llm, last=bullet_point_prompt | llm)

print("Pipeline créé : Résumé -> Points de puce.")

In [ ]:
import time

# 4. Exécutez un court paragraphe et inspectez les balles.
long_text = """L'énergie solaire est une source d'énergie renouvelable qui utilise la lumière du soleil pour générer de l'électricité. Elle est de plus en plus populaire en raison de son faible impact environnemental et de son potentiel à réduire la dépendance aux combustibles fossiles. Les panneaux solaires, également appelés modules photovoltaïques, sont les principaux composants des systèmes d'énergie solaire. Ils sont constitués de cellules photovoltaïques qui convertissent directement la lumière du soleil en électricité. L'installation de systèmes solaires peut être coûteuse initialement, mais elle offre des économies substantielles sur les factures d'électricité à long terme. De plus, de nombreux gouvernements offrent des incitations pour encourager l'adoption de l'énergie solaire. Cependant, l'intermittence du soleil et le stockage de l'énergie restent des défis à surmonter."

print("Texte original à traiter:\n")
print(long_text)

print("\nExécution du pipeline...")
start_time = time.time()

bullet_points_result = pipeline.invoke({"text": long_text})

end_time = time.time()
print(f"Pipeline terminé en {end_time - start_time:.2f} secondes.\n")

print("Points de puce générés:\n")
print(bullet_points_result)

## Part 4 (Bonus) : Ajouter une petite chaîne de conversation

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import BufferMemory

# Initialize BufferMemory
memory = BufferMemory()

# Initialize ConversationChain with the existing LLM and the memory
conversation_chain = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True # Set to True to see the internal workings
)

print("ConversationChain with BufferMemory initialized.")

In [ ]:
# Premier tour de conversation : l'utilisateur salue
print("\n--- Premier tour ---")
response1 = conversation_chain.predict(input="Bonjour, comment ça va ?")
print(f"Réponse du modèle : {response1}")

In [ ]:
# Deuxième tour de conversation : l'utilisateur demande un suivi, observant la mémoire
print("\n--- Deuxième tour ---")
response2 = conversation_chain.predict(input="Est-ce que tu te souviens de ma question précédente ?")
print(f"Réponse du modèle : {response2}")

## Livrables : Observations

### Réécriture de texte (Partie 2)
*   **Latence:** [Décrivez la latence observée lors de l'exécution de la réécriture de texte. Par exemple : "La latence était d'environ X secondes, ce qui est acceptable pour de petits modèles sur CPU."]
*   **Qualité:** [Commentez la qualité de la réécriture. "Le modèle a réussi à simplifier le texte, mais il a parfois raccourci l'information de manière significative." ou "La simplification était très bonne et conservait le sens original."]
*   **Bizarreries/Anomalies:** [Mentionnez toute bizarrerie ou comportement inattendu. Par exemple : "Le modèle a parfois ajouté des informations non présentes dans le texte original." ou "La longueur de la sortie était parfois incohérente."]

### Pipeline de résumé et de points de puce (Partie 3)
*   **Latence:** [Indiquez la latence pour le pipeline en deux étapes. "Le pipeline a pris environ Y secondes, la plupart du temps étant consacrée au résumé initial."]
*   **Qualité du résumé:** [Évaluez la qualité du résumé en une phrase. "Le résumé était concis et précis, capturant l'idée principale."]
*   **Qualité des points de puce:** [Évaluez la conversion en points de puce. "Les points de puce étaient pertinents mais n'étaient pas toujours exactement au nombre de 3." ou "Les points de puce étaient bien formatés et pertinents."]
*   **Bizarreries/Anomalies:** [Tout problème rencontré. "Parfois, le modèle répétait des informations dans différents points de puce." ou "La transition entre le résumé et les points de puce était fluide."]

### Chaîne de conversation (Partie 4 - Bonus)
*   **Rétention de mémoire:** [Décrivez comment le modèle a géré la mémoire. "Le modèle a bien retenu le contexte de la salutation initiale, ce qui a permis une conversation cohérente." ou "La mémoire était limitée et le modèle a eu du mal à se souvenir précisément de la question précédente."]
*   **Qualité de la conversation:** [Commentez la fluidité et la pertinence des réponses. "Les réponses étaient naturelles et semblaient tenir compte du contexte." ou "Les réponses étaient un peu génériques malgré la mémoire."]
*   **Bizarreries/Anomalies:** [Tout comportement étrange. "Le modèle a parfois généré des réponses hors sujet." ou "Le `verbose=True` a été utile pour comprendre le fonctionnement interne de la chaîne."]

N'oubliez pas d'exécuter les cellules et de remplir ces sections d'observations avec vos propres résultats !